# 02 — Transfer learning strategies

This notebook introduces basic transfer learning strategies. Naive fine-tuning, freezing, and a simple LoRA demonstration. 
Multiheaded training is in notebook 3.

## 1. Base setup

We first load a base model `base_model.pt` that was saved from prior training. 

Then we set up in the HF training data / dataloaders etc and set them up in the same way as [tutorial 01](01_base_model_training.ipynb).

In [ ]:
# Imports 
import json
import random
from pathlib import Path

import numpy as np
import torch
import ase.io
from ase.data import chemical_symbols

from mace.data.atom_data_loader import AtomDataLoaderBuilder
from mace.modules.loss import InvariantsWeightedEnergyForcesNacsDipoleLoss
from mace.training.trainer import Trainer
from mace.training import NaiveStrategy, FreezeStrategy, LoRAStrategy

# Set up
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_default_dtype(torch.float64)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load in base model from path
base_model_path = "<insert full model file path>"
base_model = torch.load(base_model_path, map_location="cpu", weights_only=False)

# Load in training and validation data
train_atoms = ase.io.read("<insert target training file path>", index=":")
valid_atoms = ase.io.read("<insert target validation file path>", index=":")

# Typically specify new E0s for higher fidelity data
tl_e0s = {
    "H": -15.35, "C": -1032.08
}
data_builder = AtomDataLoaderBuilder(
    cutoff=float(base_model.r_max), 
    E0s=tl_e0s
)

# Same set up as before
train_loader = data_builder.load(train_atoms, batch_size=10, shuffle=True, seed=seed)
valid_loader = data_builder.load(valid_atoms, batch_size=10, shuffle=False)

loss_fn = InvariantsWeightedEnergyForcesNacsDipoleLoss(
    energy_weight=1.0, 
    forces_weight=1.0, 
    nacs_weight=1.0,
).to(device)

trainer = Trainer(
    device=device, 
    max_epochs=100, 
    optimiser_lr=1e-3,
    ema_decay=0.99, 
    restore_best=True,
)

# Make directory for the TL, to save checkpoints and logs etc
output_dir = Path("<insert new transfer run directory>")
output_dir.mkdir(parents=True, exist_ok=False)

## 2. Strategy class overview

A strategy is a class that is built for modifying a base model so that it is suitable to re-train for transfer learning. The various strategies can be initialised with different settings. The strategies are then applied to the model by calling the `apply` function. Every strategy class has that function, which takes in a model and returns a modified model. 

An example call would be
```python
transfer_model = strategy.apply(base_model)
```

There are 4 main strategies implemented so far
1) Naive Strategy: Just deep copies the model
2) Freeze Strategy: Freezes certain weights 
3) LoRA: Injects LoRA to specified layers 
4) Multiheaded training (with correction route): Covered in tutorial 3

## 3. Naive training

`NaiveStrategy` simply makes a deep copy of the base model. The copy keeps the learned weights and architecture, while the original model stays unchanged.

Pass `naive_model` to `trainer.train_model()` as in tutorial 01 to continue training on the new data. This starts a new training run from the existing weights, not from random weights. This is realistically probably gonna be the most successful strategy. Since it just re trains everything from scratch. 

In [ ]:
# Example call with naive strategy, just a deep copy only so original model is not modified
naive_strategy = NaiveStrategy()
naive_model = naive_strategy.apply(base_model)

## 4. Freeze strategy
`FreezeStrategy` copies the model and freezes the selected parameters. So far when we tested this it was generally less robust and successful. 

### Input format and default

`frozen_layers` is used to specify which layers to keep frozen, and takes a **list of strings** as input. Tuples are also accepted. Each entry can be an alias, an exact module path, or an exact parameter name.
By default, if left unspecified, the Freeze strategy will freeze the full graph only.

| Call | What it freezes |
|---|---|
| `FreezeStrategy()` | Shared graph modules `full_graph` (Default settings)|
| `FreezeStrategy(frozen_layers=["encoder"])` | Encoders only |
| `FreezeStrategy(frozen_layers=[])` | No additional layers frozen |


### Available aliases

The following are the available aliases that can be specified under the freeze strategy 

| Alias | Parameters selected |
|---|---|
| `full_graph` | Node embedding, radial embedding, interactions, and products |
| `node` | Node embedding |
| `interactions` | All interaction blocks |
| `products` | All product blocks |
| `encoder` | Energy encoders in every head |
| `decoder` | Energy decoders in every head |
| `invariant_readouts` | Scalar energy-latent readouts in every head |
| `nac_readouts` | NAC readouts in every head |
| `soc_readouts` | SOC readouts in every head |
| `readouts` | Invariant, NAC, and SOC readouts in every head |
| `all_heads` | All modules inside every autoencoder head |
| `head0` | All modules inside the first autoencoder head |

`readouts` excludes the encoders and decoders.

### Exact paths and parameter names

For more specific selections, you can also specify exact paths and parameter names to freeze. For example, 

- `autoencoder_heads.0.perm_encoder` for only the first head's encoder.
- `interactions.0` for only the first interaction block.
- An exact name from `model.named_parameters()` for a single parameter tensor.

Names must be nonempty and cannot be duplicated. Each selection must match at least one currently trainable parameter exactly, and at least one parameter must remain trainable. For example, selecting `soc_readouts` raises an error if the model has no SOC readouts.

In [ ]:
# This example freezes the graph, encoder and invariant readouts 
# Those weights in transfer_model will now be frozen and not updated during training 
freeze_strategy = FreezeStrategy(
    frozen_layers=[
        "full_graph", 
        "encoder", 
        "invariant_readouts"
    ]
)

transfer_model = freeze_strategy.apply(base_model)

## 5. LoRA

`LoRAStrategy` copies the model and injects a low-rank adapter to specific layers. The original weights are left frozen. The selected LoRA layers should be specified when the strategy is initialised, similar to the freeze strategy. This implementation of LoRA follows the original MACE implementation, all the variouss linear or equivariant layers are the same. 

### Input format and defaults

- `rank=4` is the default adapter size. It must be a positive integer; for equivariant layers, it sets the channels per shared feature type.
- `alpha=1.0` is the default scaling value. It must be finite and greater than zero. The update is scaled by `alpha / rank`.
- `lora_layers` must be a nonempty **list of nonempty strings**. Tuples are also accepted. Same as how the freezing layers were specified. What's different is that there is no default layer selection: `LoRAStrategy()` without layers raises an error. 

### What can be selected?

Use the aliases listed in section 4, or an exact module path from `model.named_modules()`. 

The key differences from freezing are:

- **Radial embedding is excluded from LoRA's `full_graph`.** That alias covers node embedding, interactions, and products. Selecting `"radial_embedding"` directly fails because it currently has no compatible linear layers.
- **Individual parameter names are not accepted.** Select `"autoencoder_heads.0.invariant_readouts.0.linear"`, not `"autoencoder_heads.0.invariant_readouts.0.linear.weight"`. Freezing can select a weight tensor; LoRA needs the module containing it.

In [ ]:
# Here is an example of how to use strategies together 

# First the FreezeStrategy is used and applied to the model to freeze the graph weights
strat_1 = FreezeStrategy(
    frozen_layers=["full_graph"]
)

lora_input_model = strat_1.apply(base_model)

# Then we further want to apply LoRA to the decoder, so that only a low rank adaptation is learned
# Define LoRA strategy, specify the rank, alpha and layers
strat_2 = LoRAStrategy(
    rank=4, 
    alpha=1.0,
    lora_layers=["decoder"]
)

# Then apply to the final model
lora_model = strat_2.apply(lora_input_model)

# lora_model is now a model with frozen graph weights and LoRA applied to the decoder

## 6. Additional notes
Currently through the brief tests that we have run, the transfer learning strategies for freezing generally makes the model worse. Not too surprising there though. It does train slightly faster but so far not sure if it generalises better or not. 
LoRA has not been tested yet, but might be interesting to try.